# Assignment 10 — Healthcare Cognitive Assistant (Heart Health)

**Jiya Mahajan | Roll No. 1024170296 | Batch 3Q31 / 3Q3A**

## Q1

The even roll number selects the Heart Health Information Assistant. This notebook provides general educational information, not diagnosis or personal treatment advice.

In [ ]:
import re, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

examples = {
"blood_pressure": ["How can I keep track of my blood pressure?", "What does systolic pressure mean?", "What is diastolic pressure?", "How should I record home readings?", "When is a good time to measure blood pressure?", "What is a blood pressure log?", "How often should I review readings with a clinician?", "What can affect a blood pressure reading?"],
"heart_symptoms": ["What are common signs of a heart attack?", "Why might someone feel heart palpitations?", "What can shortness of breath mean?", "How do I describe chest discomfort to a doctor?", "What should I do about recurring dizziness?", "Can fatigue be related to heart health?", "When should I discuss swelling with a clinician?", "What symptoms should I write down?"],
"lifestyle": ["What foods support heart health?", "How can I reduce salt in meals?", "What kind of activity is good for the heart?", "How can I start walking regularly?", "Why does sleep matter for heart health?", "What are ways to manage stress?", "How can I build a balanced meal?", "What habits support cardiovascular wellness?"],
"medications": ["What questions should I ask about my heart medicine?", "How should I store prescribed medication?", "What should I do if I have a side effect?", "Why is taking medicine as prescribed important?", "Can a pharmacist explain drug interactions?", "How do I keep a current medication list?", "What should I ask before changing a medicine?", "Where can I learn about a prescription?"],
"appointments": ["What should I bring to a cardiology appointment?", "What questions can I ask my doctor about cholesterol?", "How can I prepare for a heart checkup?", "What is an ECG generally used to check?", "What does a cholesterol test measure?", "How can I find a heart health specialist?", "What information should I share at a visit?", "How do I organize questions for my care team?"]
}
records=[(query,intent) for intent,queries in examples.items() for query in queries]
data=pd.DataFrame(records,columns=["query","intent"])
print("Labeled queries:",len(data),"categories:",data.intent.nunique())
display(data.groupby("intent").size().rename("count").to_frame())

## Q2

In [ ]:
taxonomy = {
    "measurement": ["blood_pressure", "cholesterol_screening"],
    "wellbeing": ["heart_symptoms", "risk_factors"],
    "daily_care": ["nutrition", "physical_activity"],
    "clinical_support": ["medications", "appointments"],
}
taxonomy

## Q3

In [ ]:
concepts = {
    "Person":"individual seeking heart health information", "Heart":"organ circulating blood", "BloodPressure":"force of blood against artery walls", "Systolic":"pressure during a heartbeat", "Diastolic":"pressure between heartbeats", "Cholesterol":"blood lipid measured in screening", "Medication":"prescribed medicine", "Activity":"physical movement", "Meal":"food consumed", "Clinician":"qualified health professional", "Appointment":"scheduled clinical visit", "Symptom":"reported physical experience"
}
relationships = [
    ("Person","measures","BloodPressure"), ("BloodPressure","includes","Systolic"),
    ("BloodPressure","includes","Diastolic"), ("Clinician","reviews","BloodPressure"),
    ("Clinician","orders","Cholesterol"), ("Person","reports","Symptom"),
    ("Symptom","may prompt discussion with","Clinician"), ("Person","may be prescribed","Medication"),
    ("Person","participates in","Activity"), ("Activity","supports","Heart"),
    ("Meal","can influence","Cholesterol"), ("Appointment","connects Person with","Clinician")
]
print("Concepts:",len(concepts),"Relationships:",len(relationships))
display(pd.DataFrame(relationships,columns=["subject","relation","object"]))

## Q4

In [ ]:
import nltk
for resource in ["stopwords", "wordnet", "omw-1.4"]: nltk.download(resource, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words=set(stopwords.words("english")); lemmatizer=WordNetLemmatizer()
def preprocess(text):
    tokens=re.findall(r"[a-z]+",text.lower())
    return " ".join(lemmatizer.lemmatize(token) for token in tokens if token not in stop_words)
X_train,X_test,y_train,y_test=train_test_split(data.query,data.intent,test_size=0.25,random_state=42,stratify=data.intent)
model=Pipeline([("tfidf",TfidfVectorizer(preprocessor=preprocess,ngram_range=(1,2))), ("classifier",LogisticRegression(max_iter=1000,class_weight="balanced"))])
model.fit(X_train,y_train); predictions=model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,predictions))
precision,recall,f1,_=precision_recall_fscore_support(y_test,predictions,average="macro",zero_division=0)
print("Macro precision:",precision,"Macro recall:",recall,"Macro F1:",f1)
print(classification_report(y_test,predictions,zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test,predictions,labels=model.classes_),index=model.classes_,columns=model.classes_))

Preprocessing lowercases, tokenizes, removes stopwords, and lemmatizes. TF-IDF features train a Logistic Regression classifier. Scores from this small illustrative corpus are a demonstration, not validated clinical performance.

## Q5

In [ ]:
knowledge_base = {
    "blood_pressure": "A validated home monitor and a consistent measurement routine can help record blood pressure. Share readings with a clinician, who can interpret them in context.",
    "heart_symptoms": "Symptoms can have many causes. Discuss new or recurring symptoms with a qualified clinician. Sudden severe chest pain, trouble breathing, fainting, or stroke signs require emergency help.",
    "lifestyle": "Heart-supportive habits can include balanced meals, regular activity, adequate sleep, and avoiding tobacco. Choose changes appropriate to your situation with your care team.",
    "medications": "Use medication as prescribed. Ask a pharmacist or clinician about storage, side effects, missed doses, and interactions; do not change treatment based on this assistant.",
    "appointments": "Bring questions, medication details, and relevant readings to appointments. A clinician can explain tests and discuss personal risk factors."
}
emergency_pattern=re.compile(r"\b(chest pain|trouble breathing|unconscious|seizure|severe confusion|fainting|stroke symptoms)\b",re.I)
heart_terms={"heart","cardiac","blood pressure","cholesterol","cardiology","symptom","medicine","medication","exercise","nutrition","ecg","chest pain"}
def respond(query,threshold=0.32):
    if emergency_pattern.search(query):
        return "This may be an emergency. Contact your local emergency number now or go to the nearest emergency department. Do not wait for an online response."
    if not any(term in query.lower() for term in heart_terms):
        return "I can help with general heart health information. This question appears outside my supported topics; please contact a qualified professional or relevant service."
    probabilities=model.predict_proba([query])[0]; best_index=probabilities.argmax(); confidence=float(probabilities[best_index]); intent=model.classes_[best_index]
    if confidence<threshold:
        return "I am not confident which heart health topic you mean. Could you rephrase? For personal advice, contact your care team."
    return knowledge_base.get(intent,"I do not have supported information for that topic. Please consult your care team.")

test_queries=["How can I track blood pressure?", "What do palpitations mean?", "What foods support heart health?", "What should I ask about medicine?", "What do I bring to cardiology?", "How do I check pressure at home?", "How can I repair a bicycle?", "Can you explain this thing?", "I have chest pain and trouble breathing"]
for query in test_queries: print("Q:",query,"\nA:",respond(query),"\n")

The assistant escalates emergency terms, rejects unsupported topics, requests clarification for low-confidence classifications, and retrieves general information for supported intents. It avoids diagnosis and personalized treatment instructions.

## Q6

In [ ]:
def interactive_assistant():
    print("Heart Health Information Assistant. Type 'quit' to finish.")
    while True:
        query=input("You: ").strip()
        if query.lower() in {"quit","exit"}: break
        print("Assistant:",respond(query))

interactive_assistant()